# Доставка и оценки покупателей Olist: распределение оценок и статистические проверки

Версия 0.2. Продолжение [анализа доставки](../REPORT.md): вместо двух групп «опоздал / вовремя» и средней оценки — полная шкала 1–5, длина задержки и статистические проверки. Итоги и ограничения — в [отчёте v0.2](../REPORT_v0.2.md).

**Выборка и определения** — те же, что в версии 0.1:

- заказы со статусом `delivered` и известными фактической и предполагаемой датами доставки — 96 470;
- **опоздание** — доставка начиная со следующего календарного дня после предполагаемой даты;
- **задержка в днях** — календарная дата фактической доставки минус предполагаемая (отрицательная — доставка раньше срока);
- у заказа с несколькими отзывами берётся последний по `review_answer_timestamp`;
- **низкая оценка** — 1 или 2.

Данные читаются из PostgreSQL (схема `olist`) ролью только для чтения. Порядок подготовки базы — [docs/reproduce.md](../../../docs/reproduce.md).

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from scipy.stats import chi2_contingency, spearmanr, mannwhitneyu
from scipy.stats.contingency import association
import statsmodels.formula.api as smf

# PostgreSQL доступен через SSH-туннель на локальном порту 15432.
# Пароль роли analyst хранится в ~/.pgpass и в ноутбук не записывается.
engine = create_engine("postgresql+psycopg://analyst@127.0.0.1:15432/home_data_lab")

## 0. Выборка: повторение результата версии 0.1

In [2]:
df = pd.read_sql(
    "SELECT order_id, order_status, order_delivered_customer_date, order_estimated_delivery_date "
    "FROM olist.orders",
    engine,
)

delivered = df[
    (df['order_status'] == 'delivered')
    & (df['order_delivered_customer_date'].notna())
    & (df['order_estimated_delivery_date'].notna())
].copy()

delivered['late'] = delivered['order_delivered_customer_date'] >= (
    delivered['order_estimated_delivery_date'] + pd.Timedelta(days=1)
)

len(delivered), delivered['late'].sum(), round(delivered['late'].mean() * 100, 2)

(96470, np.int64(6534), np.float64(6.77))

In [3]:
df_reviews = pd.read_sql(
    "SELECT order_id, review_score, review_creation_date, review_answer_timestamp "
    "FROM olist.order_reviews",
    engine,
)

# Последний отзыв каждого заказа
df_reviews = (
    df_reviews
    .sort_values(by=['review_answer_timestamp'], ascending=False)
    .drop_duplicates(subset=['order_id'], keep='first')
)

df_pre = delivered.merge(df_reviews, how='left', on='order_id', validate='1:1')

# Заказы без оценки исключаются явно
df_pre['review_score'].isna().sum()

np.int64(646)

In [4]:
df_scored = df_pre.dropna(subset=['review_score']).copy()
df_scored['review_score'] = df_scored['review_score'].astype(int)
len(df_scored)

95824

Совпадает с версией 0.1: 96 470 заказов, 6 534 опоздания (6,77%), 646 заказов без оценки, 95 824 заказа с оценкой.

## 1. Распределение оценок 1–5

**Вопрос:** как распределены оценки 1–5 у опоздавших и у пришедших вовремя заказов? Доли считаются внутри группы доставки: каждый столбец долей в сумме — 100%.

In [5]:
res1 = pd.crosstab(df_scored['review_score'], df_scored['late'])
res2 = (pd.crosstab(df_scored['review_score'], df_scored['late'], normalize='columns') * 100).round(1)

result = (
    res1.merge(res2, how='inner', on='review_score')
    .rename(columns={'False_x': 'Вовремя', 'True_x': 'Опоздали',
                     'False_y': 'Доля вовремя, %', 'True_y': 'Доля опоздали, %'})
    .sort_index(ascending=False)
)
result

late,Вовремя,Опоздали,"Доля вовремя, %","Доля опоздали, %"
review_score,,,,
5,55693,1055,62.3,16.5
4,18239,649,20.4,10.2
3,7222,694,8.1,10.9
2,2369,552,2.6,8.7
1,5920,3431,6.6,53.8


**Вывод.** У опоздавших самая частая оценка — 1 (53,8%), вторая по частоте — 5 (16,5%): распределение поляризовано. У пришедших вовремя 82,6% оценок — 4 или 5. Даже у пришедших вовремя оценка 1 встречается чаще, чем 2 (6,6% против 2,6%).

## 2. Длина задержки и оценка

**Вопрос:** правда ли, что опоздавшие заказы с высокой оценкой опоздали ненадолго?

In [6]:
df_scored['delay_days'] = (
    df_scored['order_delivered_customer_date'].dt.normalize()
    - df_scored['order_estimated_delivery_date']
).dt.days

late_true = df_scored[df_scored['late']].copy()

# Медианная задержка среди всех опоздавших (в версии 0.1 — 7 дней)
late_true['delay_days'].median()

np.float64(7.0)

In [7]:
median_late_days = (
    late_true
    .groupby('review_score')
    .aggregate({'delay_days': 'median'})
    .sort_index(ascending=False)
    .rename(columns={'delay_days': 'Медиана задержки, дней'})
)
median_late_days

,"Медиана задержки, дней"
review_score,
5,3.0
4,3.0
3,5.0
2,7.0
1,8.0


In [8]:
bins = [0, 2, 5, 10, float('inf')]
labels = ['1–2', '3–5', '6–10', '11+']
late_true['group_delay'] = pd.cut(late_true['delay_days'], bins=bins, labels=labels)

late_true['group_delay'].isna().sum()

np.int64(0)

In [9]:
# Количество заказов: группа задержки × оценка
pd.crosstab(late_true['group_delay'], late_true['review_score'])

review_score,1,2,3,4,5
group_delay,,,,,
1–2,254,91,207,313,491
3–5,661,114,162,150,279
6–10,1109,157,142,86,140
11+,1407,190,183,100,145


In [10]:
# Доли оценок внутри каждой группы задержки, %
(pd.crosstab(late_true['group_delay'], late_true['review_score'], normalize='index') * 100).round(1)

review_score,1,2,3,4,5
group_delay,,,,,
1–2,18.7,6.7,15.3,23.1,36.2
3–5,48.4,8.3,11.9,11.0,20.4
6–10,67.9,9.6,8.7,5.3,8.6
11+,69.5,9.4,9.0,4.9,7.2


**Вывод.** Отчасти. Медианная задержка у опоздавших с оценкой 5 — 3 дня, с оценкой 1 — 8 дней. С ростом задержки доля оценки 1 растёт: 18,7% при задержке 1–2 дня, 48,4% при 3–5 днях и около 68–70% при задержке от 6 дней.

Короткая задержка не проходит бесследно. Даже при опоздании на 1–2 дня доля оценки 1 почти втрое выше, чем у пришедших вовремя (18,7% против 6,6%), а доля оценки 5 ниже (36,2% против 62,3%).

## 3. Отзывы, оставленные до фактической доставки

**Вопрос:** сколько покупателей ответили на анкету раньше, чем получили заказ, и меняет ли это картину? Отзыв считается оставленным до доставки, если `review_answer_timestamp` раньше `order_delivered_customer_date`.

In [11]:
df_scored['answered_before_delivery'] = (
    df_scored['order_delivered_customer_date'] > df_scored['review_answer_timestamp']
)

df_scored['answered_before_delivery'].sum()

np.int64(4653)

In [12]:
# Доля отзывов до доставки внутри каждой группы доставки, %
(pd.crosstab(df_scored['late'], df_scored['answered_before_delivery'], normalize='index') * 100).round(1)

answered_before_delivery,False,True
late,,
False,99.8,0.2
True,29.9,70.1


In [13]:
rev_late_bef_del = df_scored[df_scored['answered_before_delivery'] & df_scored['late']]
rev_late_aft_del = df_scored[~df_scored['answered_before_delivery'] & df_scored['late']]

summary = pd.DataFrame({
    'Ответ до доставки': (rev_late_bef_del['review_score'].value_counts(normalize=True) * 100).round(1),
    'Ответ после доставки': (rev_late_aft_del['review_score'].value_counts(normalize=True) * 100).round(1),
}).sort_index(ascending=False)
summary

,Ответ до доставки,Ответ после доставки
review_score,,
5,6.3,40.6
4,4.4,23.8
3,8.6,16.2
2,9.8,5.9
1,70.9,13.5


In [14]:
# Число заказов и медианная задержка в двух подгруппах опоздавших
pd.DataFrame({
    'Заказов': [len(rev_late_bef_del), len(rev_late_aft_del)],
    'Медиана задержки, дней': [rev_late_bef_del['delay_days'].median(), rev_late_aft_del['delay_days'].median()],
}, index=['Ответ до доставки', 'Ответ после доставки'])

,Заказов,"Медиана задержки, дней"
Ответ до доставки,4473,9.0
Ответ после доставки,1908,2.0


**Вывод.** До фактической доставки оставлено 4 653 отзыва (4,86% заказов с оценкой). Почти все они приходятся на опоздавшие заказы: 70,1% опоздавших против 0,2% пришедших вовремя. Возможная причина — анкета рассылается около предполагаемой даты доставки; в этом анализе это не проверялось.

Такие отзывы не исключаются. Это реакция на саму задержку: покупатель оценивает то, что заказ ещё не пришёл. Среди них 70,9% оценок 1.

Связь опоздания с низкой оценкой сохраняется и у тех, кто ответил после получения заказа (1 908 опоздавших), но слабее: оценка 1 — 13,5% против 6,6% у пришедших вовремя, оценка 5 — 40,6% против 62,3%. Момент ответа и длина задержки идут вместе: медианная задержка у ответивших до доставки — 9 дней, после — 2 дня. Поэтому это сравнение не разделяет, что сильнее связано с оценкой: момент ответа или длина ожидания.

## 4. Доверительные интервалы для доли оценки 1

**Вопрос:** может ли разница 13,5% против 6,6% объясняться случайностью выборки? Ниже 95% доверительный интервал доли: `p ± 1,96 · √(p(1 − p) / n)`.

In [15]:
def share_ci(is_event, z=1.96):
    # Доля и 95% доверительный интервал (нормальное приближение), %
    p = is_event.mean()
    n = len(is_event)
    se = np.sqrt(p * (1 - p) / n)
    return n, round(p * 100, 2), round((p - z * se) * 100, 2), round((p + z * se) * 100, 2)

on_time = df_scored[~df_scored['late']]

pd.DataFrame(
    [share_ci(rev_late_aft_del['review_score'] == 1), share_ci(on_time['review_score'] == 1)],
    columns=['n', 'Доля оценки 1, %', 'Нижняя граница, %', 'Верхняя граница, %'],
    index=['Опоздали, ответ после доставки', 'Вовремя'],
)

,n,"Доля оценки 1, %","Нижняя граница, %","Верхняя граница, %"
"Опоздали, ответ после доставки",1908,13.52,11.99,15.06
Вовремя,89443,6.62,6.46,6.78


**Вывод.** Интервалы не пересекаются: нижняя граница у опоздавших с ответом после доставки — около 12%, верхняя у пришедших вовремя — 6,78%. Разница не объясняется случайностью выборки. Интервал опоздавших шире, потому что в группе в 47 раз меньше заказов.

## 5. Связь опоздания с оценкой: χ² и V Крамера

**Вопрос:** связаны ли опоздание и оценка 1–5?

- H₀: распределение оценок одинаково у опоздавших и у пришедших вовремя.
- H₁: распределения различаются.

χ² отвечает, есть ли связь; V Крамера — насколько она сильная (0 — нет связи, 1 — полная).

In [16]:
chi2 = chi2_contingency(res1)
chi2.statistic, chi2.pvalue, chi2.dof

(np.float64(16748.577098100228), np.float64(0.0), 4)

In [17]:
# Ожидаемые количества при отсутствии связи и наблюдаемые
expected = pd.DataFrame(chi2.expected_freq, index=res1.index, columns=res1.columns).round(0)
pd.concat({'Наблюдаемые': res1, 'Ожидаемые': expected}, axis=1)

Наблюдаемые       Ожидаемые        
late               False True      False   True 
review_score                                    
1                   5920  3431    8728.0   623.0
2                   2369   552    2726.0   195.0
3                   7222   694    7389.0   527.0
4                  18239   649   17630.0  1258.0
5                  55693  1055   52969.0  3779.0

In [18]:
association(res1, method='cramer')

0.41807270126343316

**Вывод.** Связь есть: p < 0,001. V Крамера = 0,42 — между средней (0,3) и сильной (0,5) связью по ориентирам для таблицы с двумя группами. Если бы связи не было, у опоздавших ожидалось бы 623 оценки 1, а их 3 431 — в 5,5 раза больше; оценок 5 ожидалось 3 779, а их 1 055 — в 3,6 раза меньше.

При 95 тыс. заказов почти любая разница даёт маленькое p-value, поэтому для вывода важнее V и сами количества.

## 6. Дни отклонения от предполагаемой даты и оценка: корреляция Спирмена

**Вопрос:** связаны ли дни отклонения от предполагаемой даты с оценкой? Спирмен работает с рангами, поэтому подходит для порядковой шкалы 1–5 с множеством одинаковых значений и для задержек с длинным хвостом.

In [19]:
spearman = pd.DataFrame(
    [
        spearmanr(df_scored['review_score'], df_scored['delay_days']),
        spearmanr(on_time['review_score'], on_time['delay_days']),
        spearmanr(late_true['review_score'], late_true['delay_days']),
    ],
    columns=['rho', 'p-value'],
    index=['Все заказы с оценкой', 'Вовремя', 'Опоздали'],
)
spearman

,rho,p-value
Все заказы с оценкой,-0.176442,0.000000e+00
Вовремя,-0.049257,3.570614e-49
Опоздали,-0.402117,1.205563e-246


**Вывод.** Да, но только после предполагаемой даты. У пришедших вовремя связи почти нет (−0,05): ранняя доставка почти не повышает оценку. У опоздавших связь средняя (−0,40): каждый лишний день задержки связан с более низкой оценкой. По всем заказам корреляция слабая (−0,18), потому что 93% заказов пришли вовремя и разбавляют связь. Все p < 0,001.

До предполагаемой даты оценка почти не меняется, после неё — падает тем сильнее, чем дольше задержка.

## 7. Оценки у опоздавших ниже? Критерий Манна–Уитни

**Вопрос:** ниже ли оценки у опоздавших? В отличие от χ², критерий учитывает порядок оценок.

- H₀: если взять по одному заказу из каждой группы, у пришедшего вовремя оценка выше с вероятностью 0,5.
- H₁: у пришедшего вовремя оценка выше с большей вероятностью.

Мера величины эффекта — `U / (n₁ · n₂)`: вероятность того, что у случайного заказа, пришедшего вовремя, оценка выше, чем у случайного опоздавшего (равные оценки считаются наполовину).

In [20]:
late_scores = df_scored.loc[df_scored['late'], 'review_score']
on_time_scores = df_scored.loc[~df_scored['late'], 'review_score']

u_stat, p_val = mannwhitneyu(on_time_scores, late_scores, alternative='greater')
u_stat / (len(on_time_scores) * len(late_scores)), p_val

(np.float64(0.8189958487323371), np.float64(0.0))

**Вывод.** Да, оценки у опоздавших заметно ниже (p < 0,001). Если взять случайный заказ, пришедший вовремя, и случайный опоздавший, в 82% случаев у первого оценка выше.

## 8. Логистическая регрессия: связь при учёте других признаков заказа

**Вопрос:** остаётся ли связь опоздания с низкой оценкой (1–2), если одновременно учесть число позиций, стоимость товаров, стоимость доставки и вес заказа?

Результат читается через отношение шансов `exp(коэффициент)`. Шансы — отношение числа «да» к числу «нет»; это не то же самое, что вероятность.

In [21]:
df_scored['low'] = (df_scored['review_score'] < 3).astype(int)
df_scored['late'] = df_scored['late'].astype(int)

# Отношение шансов вручную по таблице 2 × 2
counts = pd.crosstab(df_scored['late'], df_scored['low'])
odds = counts[1] / counts[0]
odds, odds[1] / odds[0]

(late
 0    0.102139
 1    1.660967
 dtype: float64,
 np.float64(16.261811351821216))

In [22]:
model = smf.logit('low ~ late', data=df_scored).fit()
model.summary()

Optimization terminated successfully.
         Current function value: 0.332206
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                    low   No. Observations:                95824
Model:                          Logit   Df Residuals:                    95822
Method:                           MLE   Df Model:                            1
Date:                Wed, 11 Nov 2026   Pseudo R-squ.:                  0.1319
Time:                        17:43:26   Log-Likelihood:                -31833.
converged:                       True   LL-Null:                       -36672.
Covariance Type:            nonrobust   LLR p-value:                     0.000
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.2814      0.012   -197.851      0.000      -2.304      -2.259
late           2.7888      0.028     98.535      0.000       2.733       2.844
==============================================================================
"""

In [23]:
np.exp(model.params)

Intercept     0.102139
late         16.261811
dtype: float64

In [24]:
df_items = pd.read_sql("SELECT order_id, product_id, price, freight_value FROM olist.order_items", engine)
df_products = pd.read_sql("SELECT product_id, product_weight_g FROM olist.products", engine)

df_i = (
    df_items.merge(df_products, how='inner', on='product_id')
    .groupby('order_id', as_index=False)
    .agg(
        n_items=('order_id', 'count'),
        total_price=('price', 'sum'),
        total_freight=('freight_value', 'sum'),
        total_weight=('product_weight_g', 'sum'),
    )
)

df_scored_new = df_scored.merge(df_i, how='left', on='order_id', validate='1:1')
df_scored_new[['n_items', 'total_price', 'total_freight', 'total_weight']].isna().sum()

n_items          0
total_price      0
total_freight    0
total_weight     0
dtype: int64

In [25]:
# Если вес неизвестен у всех товаров заказа, сумма даёт 0. Такие заказы исключаются.
(df_scored_new['total_weight'] == 0).sum()

np.int64(22)

In [26]:
df_scored_new = df_scored_new[df_scored_new['total_weight'] != 0]

model_2 = smf.logit(
    'low ~ late + n_items + total_price + total_freight + total_weight',
    data=df_scored_new,
).fit()
model_2.summary()

Optimization terminated successfully.
         Current function value: 0.324554
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                    low   No. Observations:                95802
Model:                          Logit   Df Residuals:                    95796
Method:                           MLE   Df Model:                            5
Date:                Wed, 11 Nov 2026   Pseudo R-squ.:                  0.1519
Time:                        17:43:28   Log-Likelihood:                -31093.
converged:                       True   LL-Null:                       -36663.
Covariance Type:            nonrobust   LLR p-value:                     0.000
=================================================================================
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -3.0133      0.023   -129.984      0.000      -3.059      -2.968
late              2.8565      0.029     99.636      0.000       2.800       2.913
n_items           0.5411      0.018     30.485      0.000       0.506       0.576
total_price       0.0001   5.16e-05      2.818      0.005    4.43e-05       0.000
total_freight     0.0021      0.001      3.238      0.001       0.001       0.003
total_weight   5.815e-07   2.61e-06      0.223      0.824   -4.53e-06    5.69e-06
=================================================================================
"""

In [27]:
np.exp(model_2.params)

Intercept         0.049128
late             17.401338
n_items           1.717923
total_price       1.000145
total_freight     1.002151
total_weight      1.000001
dtype: float64

**Вывод.** Да, связь остаётся и почти не меняется. Отношение шансов низкой оценки у опоздавших — 16,3 без других признаков и 17,4 при равных числе позиций, стоимости, доставке и весе. Шансы низкой оценки у опоздавших примерно в 16–17 раз выше. По долям это 62,4% против 9,3%, то есть примерно в 6,7 раза: шансы и вероятности различаются.

Из добавленных признаков с низкой оценкой заметнее всего связано число позиций: ×1,72 к шансам на каждую дополнительную позицию. Стоимость товаров и доставки на понятном шаге почти не меняет шансы (×1,01 на 100 реалов цены, ×1,02 на 10 реалов доставки), вес незначим (p = 0,82). Возможное объяснение роли числа позиций — товары разных продавцов приходят отдельными посылками, часть из них позже или не та; это гипотеза, она не проверялась.

Pseudo R² — 0,13 и 0,15: модель объясняет малую часть того, почему покупатели ставят низкие оценки. Товар, обслуживание, регион покупателя и продавец в модель не входят. Исключены 22 заказа с суммарным весом 0.